# Endangered Languages by Continent

Where in the world are languages most at risk? This notebook uses Glottolog's
Agglomerated Endangerment Status (AES) from `low` to count at-risk languages
per continent and sub-region.

**Requirements**
```
pip install languages-of-the-world plotly pandas
```

In [1]:
import pandas as pd
import plotly.express as px

import low

## 1 · Load the database

In [2]:
db = low.LanguagesOfTheWorld()
print(db)

LanguagesOfTheWorld(languages=7929, countries=247, continents=5, scripts=106, speaker_counts=8969, language_names=167917)


## 2 · Endangerment tiers

`low` stores Glottolog's **Agglomerated Endangerment Status (AES)** on each
`Language.endangerment`. AES harmonises assessments from the Catalogue of
Endangered Languages (ELCat), UNESCO's Atlas of the World's Languages in Danger,
Ethnologue (EGIDS), and other sources into a single six-step scale.

Official Glottolog documentation:
[GlottoScope — Agglomerated Endangerment Status](https://glottolog.org/langdoc/status)
· [AES parameter reference](https://glottolog.org/parameters/aes)

| `low` value | Glottolog label | Typical meaning |
|---|---|---|
| `not_endangered` | not endangered | Stable community of speakers; intergenerational transmission is healthy (UNESCO *safe*; EGIDS ≤ 6a). |
| `threatened` | threatened | Still spoken, but losing ground to a dominant language (UNESCO *vulnerable*; EGIDS 6b). |
| `shifting` | shifting | Children are increasingly adopting another language; the community is in transition (UNESCO *definitely endangered*; EGIDS 7). |
| `moribund` | moribund | Only the older generation still speaks the language fluently (UNESCO *severely endangered*; EGIDS 8a). |
| `nearly_extinct` | nearly extinct | A handful of elderly speakers remain (UNESCO *critically endangered*; EGIDS 8b). |
| `extinct` | extinct | No living fluent speakers (EGIDS ≥ 9). |

Languages with no Glottolog AES assessment have `endangerment = None`. In the
analysis below we treat **at-risk** as the four tiers from `threatened` through
`nearly_extinct` (excluding `not_endangered` and `extinct`).

In [3]:
ENDANGERMENT_ORDER = [
    "not_endangered",
    "threatened",
    "shifting",
    "moribund",
    "nearly_extinct",
    "extinct",
]

AT_RISK = {"threatened", "shifting", "moribund", "nearly_extinct"}

assessed = [l for l in db.languages if l.endangerment]
print(f"Languages with AES assessment: {len(assessed):,}")
print(f"At-risk (threatened → nearly extinct): {sum(1 for l in assessed if l.endangerment in AT_RISK):,}")

Languages with AES assessment: 7,524
At-risk (threatened → nearly extinct): 3,923


## 3 · Count at-risk languages per continent

Each language is counted once per continent where it is spoken
(via `Language.countries` → `Country.continent`).

In [4]:
rows = []
for lang in db.languages:
    if lang.endangerment not in AT_RISK:
        continue
    continents_seen = set()
    for country in lang.countries:
        cont = country.continent.label
        if cont in continents_seen:
            continue
        continents_seen.add(cont)
        rows.append({
            "language": lang.label,
            "part3": lang.part3,
            "endangerment": lang.endangerment,
            "continent": cont,
            "region": country.region.label,
        })

df = pd.DataFrame(rows)
df.head(10)

,language,part3,endangerment,continent,region
0,A'ou,aou,nearly_extinct,Asia,Eastern Asia
1,Abaga,abg,nearly_extinct,Oceania,Melanesia
2,Abai Sungai,abf,shifting,Asia,South-eastern Asia
3,Abar,mij,shifting,Africa,Sub-Saharan Africa
4,Abau,aau,shifting,Oceania,Melanesia
5,Abaza,abq,threatened,Europe,Eastern Europe
6,Abellen Ayta,abp,shifting,Asia,South-eastern Asia
7,Abinomn,bsa,nearly_extinct,Asia,South-eastern Asia
8,Abkhazian,abk,threatened,Asia,Western Asia
9,Abom,aob,nearly_extinct,Oceania,Melanesia


## 4 · Stacked bar chart — endangerment tier by continent

In [5]:
tier_df = df.groupby(["continent", "endangerment"], as_index=False).size()
tier_df = tier_df.rename(columns={"size": "count"})
tier_df["endangerment"] = pd.Categorical(tier_df["endangerment"], categories=ENDANGERMENT_ORDER, ordered=True)

fig = px.bar(
    tier_df,
    x="continent",
    y="count",
    color="endangerment",
    barmode="stack",
    category_orders={"endangerment": ENDANGERMENT_ORDER},
    labels={"count": "Languages", "endangerment": "AES tier"},
    title="At-Risk Languages by Continent and Endangerment Tier",
    color_discrete_sequence=px.colors.sequential.YlOrRd_r,
)
fig.update_layout(margin=dict(l=10, r=10, t=40, b=10))
fig.show()

## 5 · Sub-region breakdown

`db.regions` provides UN M49 sub-regions. Melanesia and Polynesia often
concentrate endangered languages within Oceania.

In [6]:
africa = db.continents.get("Africa")
oceania = db.continents.get("Oceania")
print(f"{africa.label}: {len(africa.countries)} countries")
print(f"{oceania.label}: {len(oceania.countries)} countries")

melanesia = db.regions.get("Melanesia")
if melanesia:
    at_risk_mel = df[df["region"] == melanesia.label]["language"].nunique()
    print(f"At-risk languages in {melanesia.label}: {at_risk_mel}")

Africa: 60 countries
Oceania: 29 countries
At-risk languages in Melanesia: 571


## 6 · Choropleth - at-risk language density per country

In [7]:
country_rows = []
for country in db.countries:
    at_risk = sum(
        1 for lang in country.languages
        if lang.endangerment in AT_RISK
    )
    country_rows.append({
        "iso2": country.code,
        "country": country.label,
        "continent": country.continent.label,
        "at_risk_count": at_risk,
    })

country_df = pd.DataFrame(country_rows).sort_values("at_risk_count", ascending=False)
country_df.head(10)

,iso2,country,continent,at_risk_count
99,ID,Indonesia,Asia,554
174,PG,Papua New Guinea,Oceania,440
103,IN,India,Asia,222
155,MX,Mexico,Americas,216
162,NG,Nigeria,Africa,207
46,CN,China,Asia,179
29,BR,Brazil,Americas,134
11,AU,Australia,Oceania,127
45,CM,Cameroon,Africa,112
230,US,United States of America,Americas,102


In [8]:
import urllib.request, csv, io

_UN_M49_CSV = (
    "https://raw.githubusercontent.com/lukes/ISO-3166-Countries-with-Regional-Codes"
    "/master/all/all.csv"
)

with urllib.request.urlopen(_UN_M49_CSV, timeout=20) as r:
    un_text = r.read().decode("utf-8")

alpha2_to_alpha3 = {
    row["alpha-2"]: row["alpha-3"]
    for row in csv.DictReader(io.StringIO(un_text))
    if row["alpha-2"] and row["alpha-3"]
}

country_df["iso3"] = country_df["iso2"].map(alpha2_to_alpha3)
map_df = country_df.dropna(subset=["iso3"])

fig2 = px.choropleth(
    map_df,
    locations="iso3",
    color="at_risk_count",
    hover_name="country",
    hover_data={"iso3": False, "iso2": False, "continent": True, "at_risk_count": True},
    color_continuous_scale="Reds",
    labels={"at_risk_count": "At-risk languages"},
    title="At-Risk Languages per Country",
    projection="natural earth",
)
fig2.update_layout(margin=dict(l=0, r=0, t=40, b=0))
fig2.show()

## 7 · Summary

In [9]:
print(df.groupby("continent")["language"].nunique().sort_values(ascending=False))
print()
print("Most at-risk countries:")
print(country_df.nlargest(10, "at_risk_count")[["country", "continent", "at_risk_count"]].to_string(index=False))

continent
Asia        1415
Africa       775
Americas     761
Oceania      734
Europe       177
Name: language, dtype: int64

Most at-risk countries:
                 country continent  at_risk_count
               Indonesia      Asia            554
        Papua New Guinea   Oceania            440
                   India      Asia            222
                  Mexico  Americas            216
                 Nigeria    Africa            207
                   China      Asia            179
                  Brazil  Americas            134
               Australia   Oceania            127
                Cameroon    Africa            112
United States of America  Americas            102
